# 코인 변경 문제

**참고: 이 문제에는 여러 가지 해결책이 있으며 기본 재귀 관련 문제를 보여주는 전형적인 문제입니다. 메모 작성 및 간단한 반복 솔루션과 관련된 더 나은 솔루션이 있습니다. 이 문제로 인해 문제가 발생하는 경우(또는 어떤 경우에는 실행하는 데 시간이 오래 걸리는 것 같은 경우) 솔루션 노트북을 확인하고 이 문제를 해결하는 다양한 방법에 대한 자세한 설명을 보려면 결론 링크를 완전히 읽으십시오!**


이 문제는 실제로 자체 [Wikipedia 항목](https://en.wikipedia.org/wiki/Change-making_problem)이 있을 정도로 일반적입니다! 문제 설명을 다시 확인해 보겠습니다.

이것은 고전적인 재귀 문제입니다. 목표 금액n**과 고유한 동전 값의 목록(배열)이 주어지면 변경 금액을 만드는 데 필요한 가장 적은 동전은 얼마입니까? 

예를 들면:

n = 10이고 동전 = [1,5,10]인 경우. 그러면 변경하는 방법에는 4가지가 있습니다.

* 1+1+1+1+1+1+1+1+1+1

* 5 + 1+1+1+1+1

* 5+5

* 10

최소 금액은 1코인입니다.

    
## 솔루션

이는 동적 프로그래밍의 가치를 보여주는 고전적인 문제입니다. 기본적인 재귀 예제를 보여주고 이것이 실제로 이 문제를 해결하는 최선의 방법이 아닌 이유를 보여 드리겠습니다.

기본 로직을 완전히 이해하려면 아래 코드의 주석을 꼭 읽어보세요!

In [ ]:
def rec_coin(target, coins):
    """
    INPUT: Target change amount and list of coin values
    OUTPUT: Minimum coins needed to make change

    Note, this solution is not optimized.
    """

    # Default to target value
    min_coins = target

    # Check to see if we have a single coin match (BASE CASE)
    if target in coins:
        return 1

    else:
        # for every coin value that is <= than target
        for i in [c for c in coins if c <= target]:
            # Recursive Call (add a count coin and subtract from the target)
            num_coins = 1 + rec_coin(target - i, coins)

            # Reset Minimum if we have a new minimum
            if num_coins < min_coins:
                min_coins = num_coins

    return min_coins

실제로 살펴보겠습니다.

In [ ]:
rec_coin(63, [1, 5, 10, 25])

이 접근 방식의 문제점은 매우 비효율적이라는 것입니다! 이 문제를 해결하려면 매우 많은 재귀 호출이 필요할 수 있으며 비표준 동전 값(1,5,10이 아닌 동전 값 등)의 경우에도 부정확합니다.

아래 그림에서 이 접근 방식의 문제점을 볼 수 있습니다.

In [ ]:
from IPython.display import Image

Image(url="http://interactivepython.org/runestone/static/pythonds/_images/callTree.png")

여기의 각 노드는rec_coin함수 호출에 해당합니다. 노드의 라벨은 현재 코인 수를 계산하는 변경 금액을 나타냅니다. 이미 해결한 값을 어떻게 다시 계산하는지 주목하세요! 예를 들어 15는 3번 호출되었습니다. 이미 수행한 함수 호출을 추적할 수 있다면 훨씬 더 좋을 것입니다.
_____
## 동적 프로그래밍 솔루션

이것이 해당 기능의 작업 시간을 줄이는 열쇠입니다. 더 나은 해결책은 과거 결과를 기억하는 것입니다. 그러면 새로운 최소값을 계산하기 전에 이미 결과를 알고 있는지 확인할 수 있습니다.

이것을 구현해보자:

In [ ]:
def rec_coin_dynam(target, coins, known_results):
    """
    INPUT: This funciton takes in a target amount and a list of possible coins to use.
    It also takes a third parameter, known_results, indicating previously calculated results.
    The known_results parameter shoud be started with [0] * (target+1)

    OUTPUT: Minimum number of coins needed to make the target.
    """

    # Default output to target
    min_coins = target

    # Base Case
    if target in coins:
        known_results[target] = 1
        return 1

    # Return a known result if it happens to be greater than 1
    elif known_results[target] > 0:
        return known_results[target]

    else:
        # for every coin value that is <= than target
        for i in [c for c in coins if c <= target]:
            # Recursive call, note how we include the known results!
            num_coins = 1 + rec_coin_dynam(target - i, coins, known_results)

            # Reset Minimum if we have a new minimum
            if num_coins < min_coins:
                min_coins = num_coins

                # Reset the known result
                known_results[target] = min_coins

    return min_coins

테스트해보자!

In [ ]:
target = 74
coins = [1, 5, 10, 25]
known_results = [0] * (target + 1)

rec_coin_dynam(target, coins, known_results)

# 솔루션 테스트

아래 셀을 실행하여 일부 테스트 사례에 대해 함수를 테스트하세요. 

**TestCoins 클래스는 두 개의 매개변수 입력, 즉 동전 목록과 대상을 사용하여 함수만 테스트한다는 점에 유의하세요**

In [ ]:
"""
RUN THIS CELL TO TEST YOUR FUNCTION.
NOTE: NON-DYNAMIC FUNCTIONS WILL TAKE A LONG TIME TO TEST. IF YOU BELIEVE YOU HAVE A SOLUTION
"""

from nose.tools import assert_equal


class TestCoins(object):
    def check(self, solution):
        coins = [1, 5, 10, 25]
        assert_equal(solution(45, coins), 3)
        assert_equal(solution(23, coins), 5)
        assert_equal(solution(74, coins), 8)

        print("Passed all tests.")


# Run Test

test = TestCoins()
test.check(rec_coin)

# 결론 및 추가 자료

숙제는 아래 링크를 읽고 링크에 설명된 비재귀적 솔루션도 구현해 보세요!

이 문제의 변형에 대한 또 다른 훌륭한 리소스를 보려면 다음 링크를 확인하세요.
[동적 프로그래밍 코인 변경 문제](http://interactivepython.org/runestone/static/pythonds/Recursion/DynamicProgramming.html)